## Example: How to use the ilastik Python API for Pixel Classification

The first version of the ilastik API allows you to predict your data with a previously trained ilastik project directly from Python.

Used data courtesy of the Gerlich Lab

In [ ]:
from ilastik.experimental.api import PixelClassificationPipeline
import numpy
from xarray import DataArray
# Add your imports here, e.g. for loading and preprocessing data
import imageio.v3 as iio
from matplotlib import pyplot as plt

In [ ]:
project_file = "pc.ilp"
pipeline = PixelClassificationPipeline.from_ilp_file(project_file)

In [ ]:
# load the image you would like to process and wrap it in an xarray.DataArray,
# providing the appropriate dimension names
image = DataArray(iio.imread("2d_cells_apoptotic_1channel.png"), dims=("y", "x"))
prediction = pipeline.get_probabilities(image)
# show the foreground channel:
plt.imshow(prediction[..., 0])

### Saving the results

The prediction is an `xarray.DataArray` with shape `(y, x, c)` where `c` is the number of label classes.
Each channel holds the probability (0–1) that a given pixel belongs to that class.

Below are a few common ways to write results to disk.

In [ ]:
# Save the full probability map as a multi-channel TIFF.
# TIFF handles float data natively and is widely supported
# by scientific imaging tools (Fiji / ImageJ, napari, etc.).
iio.imwrite("probabilities.tiff", prediction.values)

In [ ]:
# For a simple segmentation, pick the most likely class per pixel
# and save as a single-channel image.
segmentation = numpy.argmax(prediction.values, axis=-1).astype(numpy.uint8)
iio.imwrite("segmentation.tiff", segmentation)

# Quick visual check
plt.imshow(segmentation, cmap="tab10")
plt.title("Segmentation (argmax of probabilities)")
plt.colorbar(label="class")
plt.show()